# Dois alvos para o rastreamento mamográfico
O foco 1 preserva as seis avaliações originais. O foco 2 estima se houve diagnóstico de câncer de mama em até um ano após a mamografia. Os dois usam os mesmos oito atributos, incluindo a densidade mamária previamente disponível. A avaliação do radiologista e os desfechos de câncer ficam fora das entradas. A divisão por paciente reproduz o primeiro experimento. Como o teste multicategórico já foi consultado, sua comparação é exploratória.

In [1]:
from pathlib import Path
from textwrap import fill

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import sklearn
from IPython.display import display
from matplotlib.ticker import PercentFormatter
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_score,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_predict, FixedThresholdClassifier
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, recall_score, roc_auc_score, confusion_matrix
)


In [2]:
arquivo = Path("dataset/breast_cancer_screening_bcsc.csv")
if not arquivo.is_file():
    arquivo = Path("..") / arquivo

url_bcsc = (
    "https://raw.githubusercontent.com/arthurMalveste/"
    "Breast-Cancer-Analysis-and-Classification/e1f23dd7904492be4ec9bfc49e2fbbbc45b4386d/"
    "dataset/breast_cancer_screening_bcsc.csv"
)
dados = pd.read_csv(arquivo if arquivo.is_file() else url_bcsc)

idade = "Age_At_The_Time_Of_Mammography"
categoricas = [
    "Patients_BI_RADS_Breast_Density",
    "Is_Film_Or_Digital_Mammogram",
    "Comparison_Mammogram_From_Mammography",
    "Family_History_Of_Breast_Cancer",
    "Current_Use_Of_Hormone_Therapy",
    "Binary_Indicator",
    "History_Of_Breast_Biopsy",
]
entradas = [idade, *categoricas]
nomes_classes = {
    "Negative": "Negativo",
    "Benign findings": "Achados benignos",
    "Probably benign": "Provavelmente benigno",
    "Needs additional imaging": "Precisa de imagens adicionais",
    "Suspicious abnormality": "Anormalidade suspeita",
    "Highly suggestive of malignancy": "Altamente sugestivo de malignidade",
}
classes = list(nomes_classes)
X = dados[entradas]
y = dados["Radiologists_Assessment"]
pacientes = dados["Patients_Study_ID"]
assert set(y.unique()) == set(classes) and pacientes.notna().all()

contagens = y.value_counts().reindex(classes)
resumo = pd.DataFrame({
    "Exames": contagens,
    "Participação (%)": (100 * contagens / len(y)).round(3),
}).rename(index=nomes_classes)
display(resumo)

,Exames,Participação (%)
Radiologists_Assessment,,
Negativo,26031,65.081
Achados benignos,10717,26.794
Provavelmente benigno,139,0.348
Precisa de imagens adicionais,3049,7.623
Anormalidade suspeita,57,0.143
Altamente sugestivo de malignidade,5,0.013


In [3]:
semente = 42
divisao_teste = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=semente)
treino, teste = next(divisao_teste.split(X, y, pacientes))
X_treino, X_teste = X.iloc[treino], X.iloc[teste]
y_treino, y_teste = y.iloc[treino], y.iloc[teste]
pacientes_treino, pacientes_teste = pacientes.iloc[treino], pacientes.iloc[teste]
assert set(pacientes_treino).isdisjoint(pacientes_teste)
assert set(y_treino) == set(y_teste) == set(classes)

divisao_validacao = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=semente)
particoes = list(divisao_validacao.split(X_treino, y_treino, pacientes_treino))
for ajuste, validacao in particoes:
    assert set(pacientes_treino.iloc[ajuste]).isdisjoint(pacientes_treino.iloc[validacao])
    assert set(y_treino.iloc[ajuste]) == set(y_treino.iloc[validacao]) == set(classes)

display(pd.DataFrame({
    "Treino (exames)": y_treino.value_counts(),
    "Teste (exames)": y_teste.value_counts(),
}).reindex(classes).rename(index=nomes_classes))

,Treino (exames),Teste (exames)
Radiologists_Assessment,,
Negativo,20824,5207
Achados benignos,8574,2143
Provavelmente benigno,111,28
Precisa de imagens adicionais,2439,610
Anormalidade suspeita,46,11
Altamente sugestivo de malignidade,4,1


In [4]:
def construir_pipeline(classificador):
    preprocessamento = ColumnTransformer([
        ("idade", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), [idade]),
        ("categorias", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        ), categoricas),
    ])
    return make_pipeline(preprocessamento, classificador)




## Foco 1 · Ajuste com as seis classes
Comparamos a floresta original com modelos que variam o peso das classes, o tamanho mínimo das folhas e o método de agregação das árvores. Selecionamos pelo F1 macro médio em quatro partições de treino.

In [5]:
def floresta(folha, peso):
    return RandomForestClassifier(n_estimators=100, min_samples_leaf=folha,
                                  class_weight=peso, random_state=semente, n_jobs=2)

opcoes = {
    "Floresta original": floresta(10, "balanced"),
    "Floresta sem pesos": floresta(10, None),
    "Floresta com folhas maiores": floresta(60, "balanced"),
    "Floresta com pesos intermediários": floresta(15, {
        "Negative": 1, "Benign findings": 2, "Needs additional imaging": 5,
        "Probably benign": 15, "Suspicious abnormality": 25,
        "Highly suggestive of malignancy": 45,
    }),
    "Boosting": HistGradientBoostingClassifier(max_leaf_nodes=12, l2_regularization=5,
                                              early_stopping=False, random_state=semente),
}
validacao = pd.Series({
    nome: cross_val_score(construir_pipeline(alg), X_treino, y_treino,
                          cv=particoes, scoring="f1_macro").mean()
    for nome, alg in opcoes.items()
}, name="F1 macro médio na validação").sort_values(ascending=False)
display(validacao.to_frame())
vencedor_multi = validacao.idxmax()
multi = construir_pipeline(opcoes[vencedor_multi]).fit(X_treino, y_treino)
pred_multi = multi.predict(X_teste)
print(vencedor_multi, "F1 macro no teste", f1_score(y_teste, pred_multi, average="macro"))
display(pd.DataFrame(classification_report(y_teste, pred_multi, labels=classes,
    target_names=list(nomes_classes.values()), output_dict=True, zero_division=0)).T)

,F1 macro médio na validação
Floresta com pesos intermediários,0.206493
Floresta original,0.171165
Floresta com folhas maiores,0.166349
Floresta sem pesos,0.155380
Boosting,0.153174


Floresta com pesos intermediários F1 macro no teste 0.20155280620216542


,precision,recall,f1-score,support
Negativo,0.699305,0.714615,0.706877,5207.000000
Achados benignos,0.355245,0.368175,0.361595,2143.000000
Provavelmente benigno,0.000000,0.000000,0.000000,28.000000
Precisa de imagens adicionais,0.164835,0.122951,0.140845,610.000000
Anormalidade suspeita,0.000000,0.000000,0.000000,11.000000
Altamente sugestivo de malignidade,0.000000,0.000000,0.000000,1.000000
accuracy,0.573125,0.573125,0.573125,0.573125
macro avg,0.203231,0.200957,0.201553,8000.000000
weighted avg,0.562890,0.573125,0.567690,8000.000000


## Foco 2 · Diagnóstico de câncer de mama em até um ano
Comparamos cinco algoritmos pela precisão média (AP) fora da amostra nas partições do treino. AP resume a relação entre precisão e sensibilidade ao variar o limiar. Sua referência para um escore constante é a prevalência. Selecionamos o limiar que maximiza o F1 dos positivos nessas previsões de validação. Esse critério serve à demonstração acadêmica. A tabela final mostra os custos observados nesse ponto de operação.

In [6]:
diagnostico = dados["Is_Binary_Indicator_Of_Cancer_Diagnosis"].astype(int)
yt, yv = diagnostico.iloc[treino], diagnostico.iloc[teste]
assert all(yt.iloc[v].nunique() == 2 and yt.iloc[t].nunique() == 2 for t, v in particoes)
candidatos = {
    "Regressão logística": LogisticRegression(class_weight="balanced", max_iter=1000),
    "Árvore": DecisionTreeClassifier(max_depth=4, min_samples_leaf=40,
                                     class_weight="balanced", random_state=semente),
    "Floresta": floresta(40, "balanced"),
    "Boosting": HistGradientBoostingClassifier(max_leaf_nodes=8, l2_regularization=10,
                                              early_stopping=False, random_state=semente),
    "Naive Bayes": GaussianNB(),
}
escores = {}
for nome, algoritmo in candidatos.items():
    escores[nome] = cross_val_predict(construir_pipeline(algoritmo), X_treino, yt,
                                      cv=particoes, method="predict_proba")[:, 1]
ap = pd.Series({nome: average_precision_score(yt, prob) for nome, prob in escores.items()},
               name="AP na validação").sort_values(ascending=False)
display(ap.to_frame())
vencedor_bin = ap.idxmax()
prec, rec, limites = precision_recall_curve(yt, escores[vencedor_bin])
f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
limiar = limites[np.argmax(f1)]
binario = FixedThresholdClassifier(construir_pipeline(candidatos[vencedor_bin]),
                                  threshold=limiar, response_method="predict_proba").fit(X_treino, yt)
prev = binario.predict(X_teste)
prob = binario.predict_proba(X_teste)[:, 1]
vn, fp, fn, vp = confusion_matrix(yv, prev, labels=[0, 1]).ravel()
resultado = pd.Series({
    "Exames no teste": len(yv), "Positivos reais": int(yv.sum()), "Verdadeiros positivos": vp,
    "Falsos positivos": fp, "Falsos negativos": fn, "Verdadeiros negativos": vn,
    "Precisão": precision_score(yv, prev, zero_division=0), "Sensibilidade": recall_score(yv, prev),
    "AP": average_precision_score(yv, prob), "AP de referência": yv.mean(),
    "ROC AUC": roc_auc_score(yv, prob), "Limiar selecionado no treino": limiar,
})
print(vencedor_bin)
display(resultado.to_frame("Teste reservado"))

,AP na validação
Regressão logística,0.008302
Naive Bayes,0.008141
Árvore,0.007268
Floresta,0.006750
Boosting,0.006670


Regressão logística


,Teste reservado
Exames no teste,8000.000000
Positivos reais,60.000000
Verdadeiros positivos,7.000000
Falsos positivos,392.000000
Falsos negativos,53.000000
Verdadeiros negativos,7548.000000
Precisão,0.017544
Sensibilidade,0.116667
AP,0.013423
AP de referência,0.007500


## Modelos treinados e previsão individual
O arquivo binário inclui o limiar escolhido. O exemplo usa apenas as oito colunas de entrada e verifica que as previsões dos modelos recarregados coincidem com as obtidas antes da gravação.

In [7]:
pasta = (Path("..") if Path("../dataset").is_dir() else Path(".")) / "modelos"
pasta.mkdir(exist_ok=True)
exemplo = X_teste.iloc[[0]]
for nome, modelo in [("avaliacao-ajustada", multi), ("diagnostico-em-um-ano", binario)]:
    arquivo_modelo = pasta / f"{nome}.joblib"
    joblib.dump(modelo, arquivo_modelo, compress=3)
    restaurado = joblib.load(arquivo_modelo)
    assert np.array_equal(restaurado.predict(X_teste), modelo.predict(X_teste))
    print(nome, restaurado.predict(exemplo)[0])


avaliacao-ajustada Benign findings
diagnostico-em-um-ano 0


**Foco 1.** A floresta com pesos intermediários subiu de 17,12% para 20,65% de F1 macro na validação e de 16,66% para 20,16% no teste já consultado no experimento original. A leitura desse ganho é exploratória. A acurácia passou de 36,26% para 57,31%. O modelo ajustado acertou zero dos 28 exames provavelmente benignos, zero dos 11 suspeitos e zero do único altamente sugestivo. A floresta original acertou dois dos 11 suspeitos. O ganho médio concentrou-se nas classes mais frequentes. A raridade dificulta aprender e medir o desempenho nas classes menores. A base também carece das imagens e dos achados visuais usados na avaliação radiológica.

**Foco 2.** Entre cinco algoritmos, a regressão logística obteve a melhor precisão média (AP) na validação. No teste, a AP foi 1,34%, com prevalência de 0,75%, e a ROC AUC foi 0,598. O limiar escolhido pelo F1 dos positivos na validação identificou 7 dos 60 diagnósticos, deixou 53 sem sinalização e gerou 392 falsos positivos. Dos 399 alertas, 1,75% corresponderam a diagnósticos. O modelo binário responde diretamente à pergunta do trabalho, mas esse desempenho é insuficiente para justificar triagem clínica.